# Análise do Desemprego no Brasil (2015–2024)

**Disciplina:** Análise e Visualização de Dados  
**Aluno:** [Seu Nome]  
**Data:** 2024

---

## 1. Introdução ao Problema

O desemprego é um dos indicadores socioeconômicos mais sensíveis de um país. No Brasil, a taxa de desemprego oscila de forma significativa entre regiões, setores econômicos e períodos históricos.

Este projeto analisa uma base de dados com **800 registros trimestrais** (2015–2024) cobrindo **20 estados brasileiros** e **5 setores econômicos**, buscando responder:

- Qual é a evolução histórica do desemprego no Brasil?
- Quais regiões e estados concentram maior vulnerabilidade?
- Como o setor econômico influencia a precariedade do trabalho?
- Existem correlações entre desemprego, renda, vagas e inflação?

## 2. Explicação da Base de Dados

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `ano` | int | Ano de referência (2015–2024) |
| `trimestre` | int | Trimestre (1–4) |
| `data` | str | Data de competência |
| `regiao` | str | Região geográfica do Brasil |
| `uf` | str | Unidade federativa (sigla) |
| `populacao_ativa` | int | Total da população economicamente ativa |
| `empregados` | int | Total de pessoas empregadas |
| `desempregados` | int | Total de pessoas desempregadas |
| `taxa_desemprego` | float | % da PEA sem emprego |
| `renda_media` | float | Renda média mensal (R$) |
| `setor_predominante` | str | Setor econômico dominante |
| `vagas_formais` | int | Total de vagas com carteira assinada |
| `inflacao` | float | Taxa de inflação no período |
| `nivel_risco` | str | Classificação de risco (Baixo/Médio/Alto/Crítico) |

## 3. Leitura dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

df = pd.read_csv('../dados/simulacao_desemprego_brasil.csv')
print(f'Shape: {df.shape}')
df.head()

## 4. Limpeza e Preparação

In [ ]:
# Verificar tipos e nulos
print('=== Tipos de dados ===')
print(df.dtypes)
print(f'\n=== Valores nulos ===')
print(df.isnull().sum())
print(f'\nTotal de duplicatas: {df.duplicated().sum()}')

In [ ]:
# Converter coluna de data
df['data'] = pd.to_datetime(df['data'])

# Garantir tipos numéricos
cols_num = ['populacao_ativa','empregados','desempregados',
            'taxa_desemprego','renda_media','vagas_formais','inflacao']
df[cols_num] = df[cols_num].apply(pd.to_numeric, errors='coerce')

# Categoria para variáveis qualitativas
df['nivel_risco'] = pd.Categorical(
    df['nivel_risco'],
    categories=['Baixo','Médio','Alto','Crítico'],
    ordered=True
)

print('Tipos após tratamento:')
print(df.dtypes)

## 5. Engenharia de Atributos

In [ ]:
# Taxa de emprego (complemento)
df['taxa_emprego'] = 100 - df['taxa_desemprego']

# Ratio vagas / desempregados
df['ratio_vagas_desemp'] = (df['vagas_formais'] / df['desempregados']).round(4)

# Renda deflacionada (base 2015)
inflacao_2015 = df[df['ano']==2015]['inflacao'].mean()
df['renda_real'] = (df['renda_media'] / (1 + df['inflacao']/100)).round(2)

# Trimestre como período
df['periodo'] = df['ano'].astype(str) + '-T' + df['trimestre'].astype(str)

print('Novas colunas criadas:')
print(df[['taxa_emprego','ratio_vagas_desemp','renda_real','periodo']].head())

## 6. Análise Exploratória

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

In [ ]:
# Distribuição da taxa de desemprego por região
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ordem = df.groupby('regiao')['taxa_desemprego'].median().sort_values().index
sns.boxplot(data=df, x='regiao', y='taxa_desemprego',
            order=ordem, palette='Blues', ax=axes[0])
axes[0].set_title('Distribuição da Taxa de Desemprego por Região', fontsize=13, pad=12)
axes[0].set_xlabel('Região')
axes[0].set_ylabel('Taxa de Desemprego (%)')
axes[0].tick_params(axis='x', rotation=15)

# Violin
sns.violinplot(data=df, x='regiao', y='taxa_desemprego',
               order=ordem, palette='muted', inner='quartile', ax=axes[1])
axes[1].set_title('Violinplot — Taxa por Região', fontsize=13, pad=12)
axes[1].set_xlabel('Região')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../imagens/boxplot_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evolução temporal
serie = df.groupby('ano').agg(
    taxa=('taxa_desemprego','mean'),
    renda=('renda_media','mean'),
    inflacao=('inflacao','mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(serie['ano'], serie['taxa'], 'o-', color='#2563eb',
         linewidth=2.5, markersize=7, label='Taxa de Desemprego (%)')
ax2.plot(serie['ano'], serie['renda'], 's--', color='#16a34a',
         linewidth=2, markersize=6, label='Renda Média (R$)')

ax1.axvspan(2019.6, 2021.4, alpha=0.12, color='red', label='Pandemia')
ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Taxa de Desemprego (%)', color='#2563eb', fontsize=11)
ax2.set_ylabel('Renda Média (R$)', color='#16a34a', fontsize=11)
ax1.set_title('Evolução da Taxa de Desemprego e Renda Média (2015–2024)',
              fontsize=13, pad=12)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.savefig('../imagens/serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap taxa de desemprego por UF e Ano
pivot = df.groupby(['uf','ano'])['taxa_desemprego'].mean().unstack('ano').round(1)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Taxa (%)'})
ax.set_title('Taxa de Desemprego por Estado e Ano (%)', fontsize=13, pad=12)
ax.set_xlabel('Ano')
ax.set_ylabel('UF')
plt.tight_layout()
plt.savefig('../imagens/heatmap_uf_ano.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. KPIs

In [ ]:
kpis = {
    'Taxa Média de Desemprego (%)': df['taxa_desemprego'].mean().round(2),
    'Renda Média Nacional (R$)': df['renda_media'].mean().round(2),
    'Total Desempregados (acumulado)': df['desempregados'].sum(),
    'Total Vagas Formais (acumulado)': df['vagas_formais'].sum(),
    'Inflação Média (%)': df['inflacao'].mean().round(2),
    'Estado com maior desemprego médio': df.groupby('uf')['taxa_desemprego'].mean().idxmax(),
    'Estado com menor desemprego médio': df.groupby('uf')['taxa_desemprego'].mean().idxmin(),
    'Ano de pico do desemprego': int(df.groupby('ano')['taxa_desemprego'].mean().idxmax()),
    'Correlação desemprego x inflação': df['taxa_desemprego'].corr(df['inflacao']).round(3),
}

for k, v in kpis.items():
    print(f'  {k}: {v}')

## 8. Gráficos

In [ ]:
# Gráfico de barras — Top 10 UFs por taxa de desemprego
top_uf = (df.groupby('uf')['taxa_desemprego']
          .mean().sort_values(ascending=False)
          .head(10).reset_index())

fig, ax = plt.subplots(figsize=(11, 5))
cores = ['#ef4444' if v > 12 else '#f97316' if v > 10 else '#3b82f6'
         for v in top_uf['taxa_desemprego']]
bars = ax.barh(top_uf['uf'][::-1], top_uf['taxa_desemprego'][::-1], color=cores[::-1])
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
ax.set_xlabel('Taxa Média de Desemprego (%)', fontsize=11)
ax.set_title('Top 10 Estados — Maior Taxa Média de Desemprego', fontsize=13, pad=12)
ax.set_xlim(0, 16)
plt.tight_layout()
plt.savefig('../imagens/top10_uf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de correlação
numericas = ['taxa_desemprego','renda_media','vagas_formais','inflacao',
             'populacao_ativa','empregados','desempregados']
corr = df[numericas].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Matriz de Correlação de Pearson', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('../imagens/correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico radar — Setor × Indicadores
setor_stats = df.groupby('setor_predominante').agg(
    taxa=('taxa_desemprego','mean'),
    renda=('renda_media','mean'),
    vagas=('vagas_formais','mean'),
).round(2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
indicadores = ['taxa', 'renda', 'vagas']
titulos = ['Taxa de Desemprego (%)', 'Renda Média (R$)', 'Vagas Formais (média)']
paleta = sns.color_palette('Set2', len(setor_stats))

for i, (col, titulo) in enumerate(zip(indicadores, titulos)):
    ax = axes[i]
    bars = ax.bar(setor_stats.index, setor_stats[col],
                  color=paleta, edgecolor='white')
    ax.bar_label(bars,
                 labels=[f'{v:,.0f}' for v in setor_stats[col]],
                 padding=3, fontsize=9)
    ax.set_title(titulo, fontsize=11, pad=10)
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Comparativo por Setor Econômico', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../imagens/comparativo_setor.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Interpretação dos Resultados

### Padrão Regional
A análise evidencia uma **forte polarização regional** no mercado de trabalho brasileiro. O Nordeste concentra as maiores taxas de desemprego — CE (13,65%), PB (13,47%) e BA (13,35%) —, enquanto a região Sul apresenta os melhores resultados, com PR (6,71%), SC (6,75%) e RS (6,92%).

### Impacto da Pandemia
Os anos 2020 e 2021 foram os mais críticos do período analisado, com taxa média nacional de 11,35% e 11,45% respectivamente, superando a média histórica em aproximadamente 1,4 ponto percentual.

### Análise Setorial
A Construção Civil e o Comércio lideram os índices de desemprego setorial, enquanto o setor de Serviços e Agropecuária apresentam renda média ligeiramente superior. A variação entre setores é pequena (~0,25 pp), sugerindo que o desemprego tem caráter mais estrutural do que setorial.

### Correlações
A correlação entre inflação e desemprego é **fraca** (r ≈ 0,06), indicando que a inflação isolada não explica a variação do desemprego. Por outro lado, vagas formais e renda média apresentam correlação positiva moderada, confirmando que regiões com mais formalização geram maior renda média.

## 10. Conclusão

Este projeto demonstrou como transformar uma base de dados de simulação em um produto analítico completo, abrangendo desde a limpeza de dados até a publicação de um dashboard interativo.

**Principais achados:**

1. O desemprego no Brasil é predominantemente **estrutural e regional**,    não respondendo de forma linear a variáveis como inflação.
2. A **pandemia de COVID-19** foi o principal choque exógeno do período,    elevando o desemprego em todos os estados.
3. A **geração de vagas formais** é o vetor mais forte de melhoria de renda    e redução do desemprego nas regiões analisadas.
4. Políticas de empregabilidade devem ser **regionalmente diferenciadas**,    com foco prioritário no Nordeste e nas populações em setores de Construção    e Comércio.

---

*Notebook desenvolvido para avaliação G2 · Análise e Visualização de Dados*